# MagicKey
> Passwordless auth combining magic links and passkeys

Passwords are a hassle. MagicKey gives your FastHTML app passwordless auth by combining **magic links** (one-time URLs sent to email) with **passkeys** (device-stored cryptographic credentials with optional biometric verification). Magic links handle sign-up and device recovery; passkeys make returning logins instant and phishing-resistant.

In [ ]:
#| hide
from nbdev.showdoc import show_doc

## Quick Start

Subclass `MagicKey`, override five storage methods, provide a `send_email` callable, and define two routes (`/login` and `/setup_passkey`). Below we show a minimal working app.

First, we subclass MagicKey and override the five storage methods:

In [ ]:
from fasthtml.common import *
from fasthtml.magickey import MagicKey

db = database(':memory:')
class User: id:int; email:str
class Passkey: id:str; user_id:int; public_key:bytes; sign_count:int
users = db.create(User)
passkeys = db.create(Passkey)

class Auth(MagicKey):
    def get_user_id(self, email):
        res = users(where="email = ?", where_args=[email])
        if res: return res[0].id
        return users.insert(User(email=email)).id
    def has_passkey(self, email):
        return bool(passkeys(where="user_id = ?", where_args=[self.get_user_id(email)]))
    def get_passkey(self, cred_id):
        try: r = passkeys[cred_id]
        except NotFoundError: return None
        return dict(email=users[r.user_id].email, public_key=r.public_key, sign_count=r.sign_count)
    def save_passkey(self, cred_id, email, public_key, sign_count):
        uid = self.get_user_id(email)
        passkeys.insert(Passkey(id=cred_id, user_id=uid, public_key=public_key, sign_count=sign_count))
    def update_passkey(self, cred_id, sign_count):
        passkeys.update(Passkey(id=cred_id, sign_count=sign_count))


Second, we create the app and wire up MagicKey. `send_email` receives `(email, url)` — in production you'd send a real email; here we just display the link:

In [ ]:
app, rt = fast_app()
def fake_send_email(email, url): return P(f'Magic link for {email}: ', A(url, href=url))
mk = Auth(app, send_email=fake_send_email, public_origin='http://localhost:5001')

`public_origin` is the full origin URL used for magic link URLs and WebAuthn verification (e.g. `'https://app.example.com'` in production). MagicKey automatically adds beforeware that redirects unauthenticated users to `/login`, and registers all backend routes for magic link and passkey handling.

Third, we add the `/login` and `/setup_passkey` routes:

In [ ]:
@rt
def index(auth):
    u = users[auth]
    return P(f'Hello {u.email}!'), A('Log out', href='/logout')

@rt
def login(error: str=None):
    errmsg = P(error.replace('_', ' ').title(), style='color:red') if error else ''
    return Titled('Sign In', errmsg,
                Button('Sign in with Passkey', hx_post='/request_passkey_auth', target_id='scripts'),
                Hr(),
                Form(method='post', action='/send_magic_link')(
                    Input(name='email', type='email', placeholder='you@example.com'),
                    Button('Send Magic Link')),
                Div(id='scripts'))

@rt
def setup_passkey(): return Titled('Set Up Passkey',
                            P('Set up a passkey for faster logins next time?'),
                            Button('Register Passkey', hx_post='/request_passkey_reg', target_id='scripts'),
                            Form(Button('Skip'), method='post', action='/skip_passkey_reg'),
                            Div(id='scripts'))

That's it! This is enough to provide your users with email signup via a magic link and passkey login.

## In Depth

### The Authentication Flow

Here's how the three main scenarios play out:

**First visit (magic link → passkey setup):**

A new user visits `/` and gets redirected to `/login` by the beforeware. They enter their email, this POSTs to `/send_magic_link`, which generates a one-time token, and calls your `send_email` with the link. 

When they click it, `/verify_magiclink` checks and marks the token as used, creates the user if new, and calls `after_magiclink_verify`. Since this user has no passkey yet, they're redirected to `/setup_passkey`, where they can choose to register their passkey via the browser's WebAuthn API prompts for biometric/PIN registration.

**Returning visit (passkey):**

The user clicks "Sign in with Passkey" on `/login`, which POSTs to `/request_passkey_auth`. The browser prompts for biometric/PIN, and the credential is verified server-side via `/verify_passkey_auth`. No email, fast login.

**Returning visit (magic link fallback):**

If the passkey doesn't work (new device, declined prompt, etc.), the user falls back to the email form. Since they already have a passkey registered, `after_magiclink_verify` sees this and logs them in directly, no passkey setup prompt this time.

In [ ]:
show_doc(MagicKey)

---

[source](https://github.com/AnswerDotAI/fasthtml/blob/main/fasthtml/magickey.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### MagicKey

```python

def MagicKey(
    app, # FastHTML app instance
    send_email, # `f(email, magic_url)` — sends the magic link
    public_origin, # Full origin URL, e.g. `https://example.com`
    rp_id:NoneType=None, # WebAuthn relying party ID (default: hostname from `public_origin`)
    rp_name:NoneType=None, # WebAuthn relying party display name (default: `rp_id`)
    skip:NoneType=None, # Routes to skip auth beforeware (default: all MagicKey routes)
    login_path:str='/login', # Login page route
    logout_path:str='/logout', # Logout route
    token_expiry:int=3600, # Magic link validity in seconds
    email_rate:str='3/m', # Per-email rate limit on magic link sends (None to disable)
    ip_rate:str='10/m', # Per-IP rate limit on magic link sends (None to disable)
    webauthn_js:NoneType=None, # Script tag or URL for SimpleWebAuthn browser JS (default: jsdelivr CDN)
):


```

*Passwordless auth combining magic links and passkeys*

A few of these parameters deserve more explanation than their docstring provides.

### `rp_id`

The WebAuthn [Relying Party](https://www.w3.org/TR/webauthn/#relying-party) identifier. This must be a domain that matches your site's origin — browsers will reject passkey operations otherwise. Defaults to the hostname extracted from `public_origin` (e.g. `'app.example.com'` from `'https://app.example.com'`).

The main reason to override this is the **subdomain pattern**: if your app runs at `app.example.com` but you set `rp_id='example.com'`, passkeys registered there will also work on `api.example.com`, `admin.example.com`, or any other subdomain. WebAuthn allows `rp_id` to be a registrable domain suffix of the current origin. So a passkey created with `rp_id='example.com'` is valid anywhere under `*.example.com`, but one created with `rp_id='app.example.com'` only works on that exact subdomain.


### `email_rate`, `ip_rate`

Rate limits on the `/send_magic_link` endpoint to prevent abuse. `email_rate` limits per recipient address (default `'3/m'` — 3 requests per minute), `ip_rate` limits per source IP (default `'10/m'`). Format is `'count/period'` where period is `(s)econds`, `(m)inutes`, `(h)ours`, or `(d)ays`. Set either to `None` to disable that limit.

### `webauthn_js`

The SimpleWebAuthn browser library used for passkey ceremonies. Defaults to a jsdelivr CDN script tag. Override this to self-host the JS or pin a specific version — pass either a Script(...) tag or a URL string. Self-hosting avoids a third-party dependency and ensures the script can't change unexpectedly.

### Sending Magic Links

In the example above we provided a fake example that just prints the email. To actually send emails you can use any of the many email libraries available in Python.

Here's one example available to apps deployed on [Pla.sh](https://pla.sh), an easy to use python web host, by the creators of FastHTML.

```py
from plash_cli.auth import send_magiclink
def send_email(email,url):
    res = send_magiclink(email,url)
    if res.status_code == 200: return P('Please check your email for your login link')
    else: return P('Something went wrong, please try again later')
mk = Auth(app, send_email=send_email, public_origin='https://example.com')
...
```

### Overridable Methods

Beyond the five required storage methods shown in the Quick Start (`get_user_id`, `has_passkey`, `get_passkey`, `save_passkey`, `update_passkey`), there are two optional methods you can override to customise the auth flow:

**`get_auth(self, user_id, session)`** — called after successful authentication (both magic link and passkey). Returns the URL to redirect to. Default: `'/'`. Override this to send users to a dashboard, onboarding page, or wherever makes sense for your app.

**`after_magiclink_verify(self, email, session, req)`** — called after a magic link token is verified. The default behaviour checks `has_passkey(email)`: if the user already has one, it logs them in directly; if not, it redirects to `/setup_passkey`. Override this to skip passkey setup entirely, add custom onboarding logic, or change the redirect flow.